In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

In [16]:

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train.head()


,Customer ID,Name,Gender,Age,Income (USD),Income Stability,Profession,Type of Employment,Location,Loan Amount Request (USD),...,Credit Score,No. of Defaults,Has Active Credit Card,Property ID,Property Age,Property Type,Property Location,Co-Applicant,Property Price,Loan Sanction Amount (USD)
0,C-36995,Frederica Shealy,F,56,1933.05,Low,Working,Sales staff,Semi-Urban,72809.58,...,809.44,0,NaN,746,1933.05,4,Rural,1,119933.46,54607.18
1,C-33999,America Calderone,M,32,4952.91,Low,Working,NaN,Semi-Urban,46837.47,...,780.40,0,Unpossessed,608,4952.91,2,Rural,1,54791.00,37469.98
2,C-3770,Rosetta Verne,F,65,988.19,High,Pensioner,NaN,Semi-Urban,45593.04,...,833.15,0,Unpossessed,546,988.19,2,Urban,0,72440.58,36474.43
3,C-26480,Zoe Chitty,F,65,NaN,High,Pensioner,NaN,Rural,80057.92,...,832.70,1,Unpossessed,890,NaN,2,Semi-Urban,1,121441.51,56040.54
4,C-23459,Afton Venema,F,31,2614.77,Low,Working,High skill tech staff,Semi-Urban,113858.89,...,745.55,1,Active,715,2614.77,4,Semi-Urban,1,208567.91,74008.28


In [34]:
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numeric_features = X_train.select_dtypes(exclude=["object"]).columns.tolist()


In [35]:
X_train = train.drop(columns=[
    "Customer ID",
    "Name",
    "Property ID",
    "Loan Sanction Amount (USD)"
])

y_train = train["Loan Sanction Amount (USD)"]

X_test = test.drop(columns=[
    "Customer ID",
    "Name",
    "Property ID"
])
X_train.dtypes


Gender                          object
Age                              int64
Income (USD)                   float64
Income Stability                object
Profession                      object
Type of Employment              object
Location                        object
Loan Amount Request (USD)      float64
Current Loan Expenses (USD)    float64
Expense Type 1                  object
Expense Type 2                  object
Dependents                     float64
Credit Score                   float64
No. of Defaults                  int64
Has Active Credit Card          object
Property Age                   float64
Property Type                    int64
Property Location               object
Co-Applicant                     int64
Property Price                 float64
dtype: object

In [36]:
numeric_features = [
    "Age",
    "Income (USD)",
    "Loan Amount Request (USD)",
    "Current Loan Expenses (USD)",
    "Dependents",
    "Credit Score",
    "No. of Defaults",
    "Property Age",
    "Property Price"
]

binary_features = [
    "Gender",
    "Income Stability",
    "Has Active Credit Card",
    "Co-Applicant"
]

categorical_features = [
    "Profession",
    "Type of Employment",
    "Location",
    "Expense Type 1",
    "Expense Type 2",
    "Property Type",
    "Property Location"
]


In [37]:

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

binary_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("bin", binary_transformer, binary_features),
    ("cat", categorical_transformer, categorical_features)
])


In [38]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

In [39]:
model.fit(X_train,y_train)


ValueError: could not convert string to float: 'F'

In [40]:
test_predictions = model.predict(X_test)

test_df["Predicted Loan Sanction Amount (USD)"] = test_predictions


ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: '?'

In [41]:
test.to_csv("loan_predictions.csv", index=False)
